In [14]:
# Import necessary libraries
import json
import os
from typing import Any, Dict

import boto3
import requests
from dotenv import load_dotenv

load_dotenv()
print("We are up and running!")

We are up and running!


AWS credentials are picked up automatically from the environment (instance profile, env vars, or `~/.aws/credentials`).

In [15]:
MODEL_ID = "eu.amazon.nova-lite-v1:0"
REGION = "eu-west-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Bedrock client ready — model: {MODEL_ID}")

Bedrock client ready — model: eu.amazon.nova-lite-v1:0


In [16]:
PROMPT = "What is the current price of Bitcoin?"

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=[
        {"role": "user", "content": [{"text": PROMPT}]}
    ],
)

print(response["output"]["message"]["content"][0]["text"])

I'm currently unable to provide real-time data as my information is not updated in real-time. To find the current price of Bitcoin, you can check reliable financial news websites, cryptocurrency exchanges (such as Coinbase, Binance, or Kraken), or use financial apps that track cryptocurrency prices. The price of Bitcoin can fluctuate frequently, so it's best to check these sources for the most up-to-date information.


In [17]:
# Inspect the full raw response from Bedrock
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "6192a9ec-c43d-44c5-acef-e60f90511d6a",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:25:08 GMT",
      "content-type": "application/json",
      "content-length": "621",
      "connection": "keep-alive",
      "x-amzn-requestid": "6192a9ec-c43d-44c5-acef-e60f90511d6a"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "I'm currently unable to provide real-time data as my information is not updated in real-time. To find the current price of Bitcoin, you can check reliable financial news websites, cryptocurrency exchanges (such as Coinbase, Binance, or Kraken), or use financial apps that track cryptocurrency prices. The price of Bitcoin can fluctuate frequently, so it's best to check these sources for the most up-to-date information."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens"

In [18]:
url = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"
response = requests.get(url)
data = response.json()
print(data)

{'symbol': 'BTCUSDT', 'price': '66190.02000000'}


In [19]:
# Define the function
def get_crypto_price(symbol: str) -> float:
    """
    Get the current price of a cryptocurrency from Binance API
    """
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    data = response.json()
    return float(data["price"])

In [20]:
price = get_crypto_price("BTCUSDT")
print(f"BTC Price in USDT: {price}")

BTC Price in USDT: 66190.01


In [21]:
# Tool definition in Bedrock Converse format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_crypto_price",
                "description": "Get cryptocurrency price in USDT from Binance",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The cryptocurrency trading pair symbol "
                                    "(e.g., BTCUSDT, ETHUSDT). "
                                    "The symbol for Bitcoin is BTCUSDT. "
                                    "The symbol for Ethereum is ETHUSDT."
                                ),
                            }
                        },
                        "required": ["symbol"],
                    }
                },
            }
        }
    ]
}

In [22]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "9c23230e-6323-45f8-bb7b-bd9a6db08f9e",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:25:09 GMT",
      "content-type": "application/json",
      "content-length": "524",
      "connection": "keep-alive",
      "x-amzn-requestid": "9c23230e-6323-45f8-bb7b-bd9a6db08f9e"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>The User wants to know the current price of Bitcoin. I can use the \"get_crypto_price\" tool to fetch this information. I need to specify the symbol for Bitcoin, which is \"BTCUSDT\".</thinking>\n"
        },
        {
          "toolUse": {
            "toolUseId": "tooluse_ohDVKr9iL9KlWYtb7sQFlM",
            "name": "get_crypto_price",
            "input": {
              "symbol": "BTCUSDT"
            }
          }
        }
      ]
    }
  },
  "stopReason": "tool_use",
  "usage": {
    "

In [23]:
# Extract the tool call from the response
assistant_message = response["output"]["message"]
tool_use_block = next(
    block for block in assistant_message["content"] if "toolUse" in block
)

tool_use_id = tool_use_block["toolUse"]["toolUseId"]
tool_name = tool_use_block["toolUse"]["name"]
tool_input = tool_use_block["toolUse"]["input"]

print(f"Tool requested: {tool_name}")
print(f"Tool input:     {tool_input}")
print(f"Tool use ID:    {tool_use_id}")

Tool requested: get_crypto_price
Tool input:     {'symbol': 'BTCUSDT'}
Tool use ID:    tooluse_ohDVKr9iL9KlWYtb7sQFlM


In [24]:
# Manually call the function with the arguments the model provided
price = get_crypto_price(**tool_input)
print(f"Result from function: {price}")

Result from function: 66193.29


In [25]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "a4b19f6c-5ee6-410e-87be-148b76afcd58",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:25:10 GMT",
      "content-type": "application/json",
      "content-length": "411",
      "connection": "keep-alive",
      "x-amzn-requestid": "a4b19f6c-5ee6-410e-87be-148b76afcd58"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>I have used the \"get_crypto_price\" tool and received the result. The current price of Bitcoin (BTCUSDT) is $66193.29.</thinking>\n\n<response>The current price of Bitcoin is $66193.29.</response>"
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 551,
    "outputTokens": 64,
    "totalTokens": 615
  },
  "metrics": {
    "latencyMs": 724
  }
}


In [26]:
import re

text = final_response["output"]["message"]["content"][0]["text"]
clean_text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

print(clean_text)

<response>The current price of Bitcoin is $66193.29.</response>
